In [1]:
# 05_german_replication.ipynb
# Generalization replication on UCI German Credit (1000 rows, 30% bad),
# which is structurally UNLIKE GMSC: 13 nominal + 7 numeric features.
# Tests whether the WoE<->stump isomorphism and the performance parity hold
# when most predictors are UNORDERED CATEGORICAL (no monotonicity to exploit).
# Reuses validated logic: credit-convention WoE, log-odds-limit stump (03 showed B==A),
# fair-coordinate bootstrap SE, repeated-split AUC with DeLong.
# Single-cell; greyscale seaborn; dpi600 png+pdf; no captions.

import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
import statsmodels.api as sm
warnings.filterwarnings('ignore')

SEED = 42; N_BOOT = 500; N_SPLIT = 100
rng = np.random.default_rng(SEED)
sns.set_theme(style='whitegrid', context='paper'); sns.set_palette('Greys')
plt.rcParams.update({'axes.edgecolor':'0.2','grid.color':'0.85','font.size':10})

ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA   = os.path.join(ROOT, 'data'); TABLES = os.path.join(ROOT,'results','tables')
FIGS   = os.path.join(ROOT, 'results','figures')
def savefig(fig,name):
    for e in ('png','pdf'): fig.savefig(os.path.join(FIGS,name+'.'+e),dpi=600,bbox_inches='tight')
    plt.close(fig)

# ===================================================================
# 0. LOAD + typing
# ===================================================================
cols = ['Status_checking','Duration','Credit_history','Purpose','Credit_amount',
        'Savings','Employment_since','Installment_rate','Personal_status_sex','Other_debtors',
        'Residence_since','Property','Age','Other_installment','Housing',
        'Existing_credits','Job','Num_liable','Telephone','Foreign_worker','target']
df = pd.read_csv(os.path.join(DATA,'german.data'), sep=' ', header=None, names=cols)
df['target'] = (df['target']==2).astype(int)                     # bad=1
TARGET='target'
cat_cols = ['Status_checking','Credit_history','Purpose','Savings','Employment_since',
            'Personal_status_sex','Other_debtors','Property','Other_installment','Housing',
            'Job','Telephone','Foreign_worker']
num_cols = ['Duration','Credit_amount','Installment_rate','Residence_since','Age',
            'Existing_credits','Num_liable']
FEATURES = cat_cols + num_cols
y = df[TARGET].values.astype(int)
TG=int((y==0).sum()); TB=int((y==1).sum())
print('German:', df.shape, '| bad rate %.3f'%y.mean(),'| %d nominal + %d numeric'%(len(cat_cols),len(num_cols)))

# ===================================================================
# 1. WoE BINNING  (credit convention: WoE=ln(bad%/good%))
#    - categorical: each level is its own bin (no monotonicity)
#    - numeric: monotonic tree binning as in GMSC/01
# ===================================================================
def woe_of(good,bad):
    gr=(good+0.5)/(TG+0.5); br=(bad+0.5)/(TB+0.5); return float(np.log(br/gr))

def num_edges(x,yv,max_leaf=5,min_frac=0.05):
    xx=x.values.reshape(-1,1).astype(float)
    if len(np.unique(xx))<=2: return []
    t=DecisionTreeClassifier(max_leaf_nodes=max_leaf,min_samples_leaf=int(min_frac*len(xx)),random_state=SEED)
    t.fit(xx,yv); thr=t.tree_.threshold[t.tree_.feature>=0]; return sorted(float(v) for v in thr)

def num_table(x,yv,edges):
    cuts=[-np.inf]+list(edges)+[np.inf]; idx=pd.cut(x,cuts,labels=False,include_lowest=True)
    rows=[]
    for b in range(len(cuts)-1):
        sel=idx==b; rows.append([b,cuts[b],cuts[b+1],int(((yv==0)&sel).sum()),int(((yv==1)&sel).sum())])
    tb=pd.DataFrame(rows,columns=['bin','lo','hi','good','bad']); tb=tb[(tb.good+tb.bad)>0].reset_index(drop=True)
    tb['bad_rate']=tb.bad/(tb.good+tb.bad); return tb

def enforce_mono(x,yv,edges):
    e=list(edges)
    while e:
        tb=num_table(x,yv,e); br=tb.sort_values('lo')['bad_rate'].values
        if len(br)<=2 or np.all(np.diff(br)>=-1e-12) or np.all(np.diff(br)<=1e-12): return e
        d=np.diff(br); s=np.sign(np.nansum(d)); e.pop(int(np.argmax(-s*d)))
    return e

woe_maps={}; iv_rows=[]
for c in cat_cols:
    recs=[]
    for i,lv in enumerate(sorted(df[c].unique())):
        sel=df[c]==lv; g=int(((y==0)&sel).sum()); b=int(((y==1)&sel).sum())
        recs.append({'bin':i,'level':lv,'good':g,'bad':b,'woe':woe_of(g,b),'n':g+b})
    woe_maps[c]={'kind':'cat','recs':recs}
    iv=sum(((r['bad']+.5)/(TB+.5)-(r['good']+.5)/(TG+.5))*r['woe'] for r in recs)
    iv_rows.append([c,'cat',iv,len(recs)])
for c in num_cols:
    edges=enforce_mono(df[c],y,num_edges(df[c],y)); tb=num_table(df[c],y,edges); recs=[]
    for _,r in tb.iterrows():
        recs.append({'bin':int(r['bin']),'lo':float(r['lo']),'hi':float(r['hi']),
                     'good':int(r['good']),'bad':int(r['bad']),'woe':woe_of(r['good'],r['bad']),'n':int(r['good']+r['bad'])})
    woe_maps[c]={'kind':'num','recs':recs}
    iv=sum(((r['bad']+.5)/(TB+.5)-(r['good']+.5)/(TG+.5))*r['woe'] for r in recs)
    iv_rows.append([c,'num',iv,len(recs)])
iv_df=pd.DataFrame(iv_rows,columns=['feature','kind','IV','n_bins']).sort_values('IV',ascending=False).reset_index(drop=True)
# IV<0.02 drop (same rule as GMSC)
IV_THRESH=0.02
kept=iv_df[iv_df.IV>=IV_THRESH]['feature'].tolist()
dropped=iv_df[iv_df.IV<IV_THRESH]['feature'].tolist()
print('\n=== German IV (sorted) ==='); print(iv_df.round(4).to_string(index=False))
print('\nKept %d / %d (IV>=%.2f). Dropped:'%(len(kept),len(FEATURES),IV_THRESH), dropped)
FEATURES=[c for c in FEATURES if c in kept]
cat_cols=[c for c in cat_cols if c in kept]; num_cols=[c for c in num_cols if c in kept]

# ===================================================================
# 2. bin-index + WoE application
# ===================================================================
def bidx_and_woe(colname):
    m=woe_maps[colname]
    if m['kind']=='cat':
        lut={r['level']:i for i,r in enumerate(m['recs'])}
        idx=df[colname].map(lut).values
        woe=np.array([r['woe'] for r in m['recs']]); w=np.array([r['n'] for r in m['recs']],float)
    else:
        recs=sorted(m['recs'],key=lambda r:r['lo']); idx=np.zeros(len(df),int)
        for p,r in enumerate(recs):
            sel=(df[colname]>r['lo']).values&(df[colname]<=r['hi']).values; idx[sel]=p
        woe=np.array([r['woe'] for r in recs]); w=np.array([r['n'] for r in recs],float)
    return idx,woe,w

Xbin={}; bin_woe0={}; bin_w={}; nbins={}
for c in FEATURES:
    idx,woe,w=bidx_and_woe(c); Xbin[c]=idx; bin_woe0[c]=woe; bin_w[c]=w; nbins[c]=len(woe)
Xw0=np.column_stack([bin_woe0[c][Xbin[c]] for c in FEATURES])

# ===================================================================
# 3. ISOMORPHISM (ideal regime): log-odds-limit stump vs WoE, + logit beta
# ===================================================================
def logodds_curve(bidx,yv,nb):
    good=np.zeros(nb); bad=np.zeros(nb); np.add.at(good,bidx[yv==0],1.0); np.add.at(bad,bidx[yv==1],1.0)
    tg=(yv==0).sum(); tb=(yv==1).sum(); return np.log((bad+0.5)/(good+0.5))-np.log(tb/tg)
def beff(s,w,wo):
    d=np.sum(w*wo**2); return float(np.sum(w*s*wo)/d) if d>0 else np.nan

logit0=sm.Logit(y,sm.add_constant(Xw0)).fit(disp=0)
beta_hat={c:logit0.params[j+1] for j,c in enumerate(FEATURES)}
beta_se ={c:logit0.bse[j+1] for j,c in enumerate(FEATURES)}
iso_rows=[]; s_hat={}
for c in FEATURES:
    s=logodds_curve(Xbin[c],y,nbins[c]); s_hat[c]=s; woe=bin_woe0[c]; w=bin_w[c]
    wm_s=np.sum(w*s)/w.sum(); wm_w=np.sum(w*woe)/w.sum()
    cov=np.sum(w*(s-wm_s)*(woe-wm_w)); vs=np.sum(w*(s-wm_s)**2); vw=np.sum(w*(woe-wm_w)**2)
    wcorr=cov/np.sqrt(vs*vw) if vs>0 and vw>0 else np.nan
    iso_rows.append([c, woe_maps[c]['kind'], nbins[c], beff(s,w,woe), wcorr, beta_hat[c], beta_se[c]])
iso_df=pd.DataFrame(iso_rows,columns=['feature','kind','n_bins','beta_eff','w_corr_s_woe','beta','beta_se'])
print('\n=== German isomorphism (ideal regime) ==='); print(iso_df.round(4).to_string(index=False))
print('mean beta_eff=%.4f | mean w_corr=%.4f'%(iso_df.beta_eff.mean(), iso_df.w_corr_s_woe.mean()))
print('  by type: cat w_corr=%.4f | num w_corr=%.4f'%(
    iso_df[iso_df.kind=='cat'].w_corr_s_woe.mean(), iso_df[iso_df.kind=='num'].w_corr_s_woe.mean()))

# ===================================================================
# 4. INFERENCE: fair-coordinate bootstrap SE (log-odds stump vs logit), N_BOOT
# ===================================================================
n=len(y)
def woe_from_counts(bidx,yv,nb,tg,tb):
    good=np.zeros(nb); bad=np.zeros(nb); np.add.at(good,bidx[yv==0],1.0); np.add.at(bad,bidx[yv==1],1.0)
    return np.log((bad+0.5)/(tb+0.5))-np.log((good+0.5)/(tg+0.5)) + 0  # placeholder
boot_logit={c:[] for c in FEATURES}; boot_stump={c:[] for c in FEATURES}
t0=time.time()
for b in range(N_BOOT):
    idx=rng.integers(0,n,n); yb=y[idx]; tgb=int((yb==0).sum()); tbb=int((yb==1).sum())
    cols=[]; wb={}
    for c in FEATURES:
        bi=Xbin[c][idx]
        good=np.zeros(nbins[c]); bad=np.zeros(nbins[c]); np.add.at(good,bi[yb==0],1.0); np.add.at(bad,bi[yb==1],1.0)
        gr=(good+0.5)/(tgb+0.5); br=(bad+0.5)/(tbb+0.5); w=np.log(br/gr)
        wb[c]=w; cols.append(w[bi]); boot_stump[c].append(np.log((bad+0.5)/(good+0.5))-np.log(tbb/tgb))
    Xwb=np.column_stack(cols)
    try:
        lb=sm.Logit(yb,sm.add_constant(Xwb)).fit(disp=0)
        for j,c in enumerate(FEATURES): boot_logit[c].append(lb.params[j+1]*wb[c])
    except Exception:
        for c in FEATURES: boot_logit[c].append(np.full(nbins[c],np.nan))
print('bootstrap %d done in %.0fs'%(N_BOOT,time.time()-t0))
se_rows=[]
for c in FEATURES:
    L=np.array(boot_logit[c]); S=np.array(boot_stump[c])
    sel=np.nanstd(L,0,ddof=1); ses=np.nanstd(S,0,ddof=1); w=bin_w[c]; wn=w/w.sum()
    se_l=float(np.sqrt(np.sum(wn*sel**2))); se_s=float(np.sqrt(np.sum(wn*ses**2)))
    se_rows.append([c,woe_maps[c]['kind'],se_l,se_s,se_s/se_l if se_l>0 else np.nan])
se_df=pd.DataFrame(se_rows,columns=['feature','kind','se_logit','se_stump','se_ratio'])
print('\n=== German SE alignment ==='); print(se_df.round(4).to_string(index=False))
print('mean SE ratio stump/logit=%.3f (cat=%.3f, num=%.3f)'%(
    se_df.se_ratio.mean(), se_df[se_df.kind=='cat'].se_ratio.mean(), se_df[se_df.kind=='num'].se_ratio.mean()))

# ===================================================================
# 5. PERFORMANCE: repeated splits, logit-WoE vs stump-boost (+DeLong)
# ===================================================================
def ks(yt,s): fpr,tpr,_=roc_curve(yt,s); return float(np.max(tpr-fpr))
def delong(yt,s1,s2):
    yt=np.asarray(yt); pos=yt==1; neg=~pos; m=pos.sum(); nn=neg.sum()
    def st(s):
        sp,sn=s[pos],s[neg]
        V10=np.array([(np.sum(sn<x)+.5*np.sum(sn==x))/nn for x in sp])
        V01=np.array([(np.sum(sp>x)+.5*np.sum(sp==x))/m for x in sn]); return V10.mean(),V10,V01
    a1,x1,y1=st(s1); a2,x2,y2=st(s2)
    S=np.cov(np.vstack([x1,x2]))/m+np.cov(np.vstack([y1,y2]))/nn
    v=S[0,0]+S[1,1]-2*S[0,1]
    if v<=0: return a1,a2,np.nan,np.nan
    z=(a1-a2)/np.sqrt(v); return a1,a2,float(z),float(2*(1-stats.norm.cdf(abs(z))))
perf=[]; dl=[]
for sp in range(N_SPLIT):
    Xtr,Xte,ytr,yte=train_test_split(Xw0,y,test_size=0.3,stratify=y,random_state=SEED+sp)
    lr=LogisticRegression(max_iter=1000).fit(Xtr,ytr); s_lr=lr.decision_function(Xte)
    gb=GradientBoostingClassifier(max_depth=1,learning_rate=0.3,n_estimators=200,random_state=SEED+sp).fit(Xtr,ytr)
    s_gb=gb.decision_function(Xte)
    for nm,s in [('logit_woe',s_lr),('stump_boost',s_gb)]:
        perf.append([sp,nm,roc_auc_score(yte,s),ks(yte,s)])
    a_gb,a_lr,z,p=delong(yte,s_gb,s_lr); dl.append([sp,a_gb-a_lr,z,p])
perf=pd.DataFrame(perf,columns=['split','model','auc','ks']); dl=pd.DataFrame(dl,columns=['split','auc_diff','z','p'])
aucl=perf[perf.model=='logit_woe'].auc; aucs=perf[perf.model=='stump_boost'].auc
w_sl=stats.wilcoxon(aucs.values,aucl.values)
print('\n=== German performance (%d splits) ==='%N_SPLIT)
print('logit-WoE  AUC=%.4f\u00b1%.4f | stump-boost AUC=%.4f\u00b1%.4f'%(aucl.mean(),aucl.std(),aucs.mean(),aucs.std()))
print('mean AUC diff (stump-logit)=%+.5f | Wilcoxon p=%.4f | DeLong frac p<0.05=%.2f'%(
    (aucs.mean()-aucl.mean()), w_sl.pvalue, (dl.p<0.05).mean()))

# ===================================================================
# 6. SAVE + FIGURES
# ===================================================================
iv_df.round(6).to_csv(os.path.join(TABLES,'05_german_iv.csv'),index=False)
iso_df.round(6).to_csv(os.path.join(TABLES,'05_german_isomorphism.csv'),index=False)
se_df.round(6).to_csv(os.path.join(TABLES,'05_german_se.csv'),index=False)
perf.round(6).to_csv(os.path.join(TABLES,'05_german_perf.csv'),index=False)

# Fig A: s(bin) vs WoE(bin) scatter, cat vs num
allw=np.concatenate([bin_woe0[c] for c in FEATURES]); alls=np.concatenate([s_hat[c] for c in FEATURES])
kinds=np.concatenate([[woe_maps[c]['kind']]*nbins[c] for c in FEATURES])
fig,ax=plt.subplots(figsize=(5.8,5.6)); lim=np.array([min(allw.min(),alls.min()),max(allw.max(),alls.max())])*1.05
ax.plot(lim,lim,color='0.0',lw=1,ls='--')
ax.scatter(allw[kinds=='num'],alls[kinds=='num'],s=48,marker='o',c='0.3',edgecolor='0.0',lw=.4,label='numeric')
ax.scatter(allw[kinds=='cat'],alls[kinds=='cat'],s=48,marker='^',c='0.65',edgecolor='0.0',lw=.4,label='categorical')
r=np.corrcoef(allw,alls)[0,1]; ax.set_title('German: s(bin) vs WoE(bin)  (r=%.3f)'%r,fontsize=10)
ax.set_xlabel('WoE(bin)'); ax.set_ylabel('cumulative stump score s(bin)')
ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect('equal'); ax.legend(frameon=True,facecolor='white',edgecolor='0.5')
savefig(fig,'05_german_s_vs_woe')

# Fig B: beta vs beta_eff
fig,ax=plt.subplots(figsize=(5.6,5.2)); b=iso_df['beta'].values; be=iso_df['beta_eff'].values
lo=min(b.min(),be.min(),0); hi=max(b.max(),be.max())*1.08
ax.plot([lo,hi],[lo,hi],color='0.0',lw=1,ls='--')
for k,mk,cl in [('num','o','0.3'),('cat','^','0.65')]:
    s=iso_df.kind==k; ax.scatter(iso_df[s]['beta'],iso_df[s]['beta_eff'],s=60,marker=mk,c=cl,edgecolor='0.0',label=k)
ax.set_xlabel(r'logistic $\beta$'); ax.set_ylabel(r'stump $\beta_{\mathrm{eff}}$')
ax.legend(frameon=True,facecolor='white',edgecolor='0.5'); savefig(fig,'05_german_beta')

# Fig C: AUC box logit vs stump
fig,ax=plt.subplots(figsize=(5.2,4.4))
data=[perf[perf.model=='logit_woe'].auc.values, perf[perf.model=='stump_boost'].auc.values]
bp=ax.boxplot(data,patch_artist=True,widths=0.6,medianprops=dict(color='0.0',lw=1.3))
for p,c in zip(bp['boxes'],['0.4','0.65']): p.set_facecolor(c); p.set_edgecolor('0.0')
ax.set_xticklabels(['logistic-WoE','stump-boost']); ax.set_ylabel('AUC')
ax.set_title('German AUC across %d splits'%N_SPLIT,fontsize=10); savefig(fig,'05_german_auc_box')

print('\nSaved figures: 05_german_s_vs_woe, 05_german_beta, 05_german_auc_box')
print('Saved tables : 05_german_iv, 05_german_isomorphism, 05_german_se, 05_german_perf')

# ===================================================================
# 7. GMSC vs GERMAN side-by-side headline
# ===================================================================
print('\n================ GENERALIZATION HEADLINE ================')
print('GERMAN (1000 rows, 30%% bad, %d nominal + %d numeric kept):'%(
    (iso_df.kind=='cat').sum(),(iso_df.kind=='num').sum()))
print('  isomorphism: mean beta_eff=%.3f, mean w_corr(s,WoE)=%.3f'%(iso_df.beta_eff.mean(),iso_df.w_corr_s_woe.mean()))
print('  SE alignment: mean ratio stump/logit=%.3f'%se_df.se_ratio.mean())
print('  performance : logit AUC=%.3f vs stump AUC=%.3f (Wilcoxon p=%.3f)'%(aucl.mean(),aucs.mean(),w_sl.pvalue))
print('Compare GMSC (02/03/04): beta_eff~1.00, iso_nomiss SE ratio~1.20, stump~logit AUC.')
print('\n05 complete.')


German: (1000, 21) | bad rate 0.300 | 13 nominal + 7 numeric

=== German IV (sorted) ===
            feature kind     IV  n_bins
    Status_checking  cat 0.6613       4
     Credit_history  cat 0.2923       5
           Duration  num 0.2664       5
            Savings  cat 0.1888       5
            Purpose  cat 0.1654      10
      Credit_amount  num 0.1510       3
           Property  cat 0.1122       4
                Age  num 0.1001       3
   Employment_since  cat 0.0861       5
            Housing  cat 0.0840       3
  Other_installment  cat 0.0586       3
Personal_status_sex  cat 0.0450       4
     Foreign_worker  cat 0.0393       2
      Other_debtors  cat 0.0312       3
   Installment_rate  num 0.0256       4
   Existing_credits  num 0.0099       2
                Job  cat 0.0090       4
          Telephone  cat 0.0063       2
    Residence_since  num 0.0016       2
         Num_liable  num 0.0000       1

Kept 15 / 20 (IV>=0.02). Dropped: ['Existing_credits', 'Job', 'Telepho


=== German isomorphism (ideal regime) ===
            feature kind  n_bins  beta_eff  w_corr_s_woe   beta  beta_se
    Status_checking  cat       4    0.9998           1.0 0.8150   0.1071
     Credit_history  cat       5    0.9998           1.0 0.6665   0.1592
            Purpose  cat      10    0.9999           1.0 1.0187   0.2120
            Savings  cat       5    0.9998           1.0 0.8114   0.2062
   Employment_since  cat       5    0.9999           1.0 0.6056   0.2887
Personal_status_sex  cat       4    0.9999           1.0 1.0158   0.4047
      Other_debtors  cat       3    0.9999           1.0 1.0232   0.4653
           Property  cat       4    0.9998           1.0 0.2437   0.2913
  Other_installment  cat       3    0.9998           1.0 0.8291   0.3379
            Housing  cat       3    0.9998           1.0 0.4343   0.3103
     Foreign_worker  cat       2    0.9998           1.0 1.0091   0.5295
           Duration  num       5    0.9998           1.0 0.5789   0.1910
      Cr

bootstrap 500 done in 2s

=== German SE alignment ===
            feature kind  se_logit  se_stump  se_ratio
    Status_checking  cat    0.1312    0.1395    1.0634
     Credit_history  cat    0.1288    0.1462    1.1353
            Purpose  cat    0.2408    0.2164    0.8986
            Savings  cat    0.1673    0.1677    1.0027
   Employment_since  cat    0.1308    0.1403    1.0730
Personal_status_sex  cat    0.1337    0.1211    0.9054
      Other_debtors  cat    0.1319    0.1074    0.8141
           Property  cat    0.1042    0.1207    1.1577
  Other_installment  cat    0.1048    0.0973    0.9286
            Housing  cat    0.0991    0.0913    0.9217
     Foreign_worker  cat    0.6306    0.1107    0.1755
           Duration  num    0.1299    0.1482    1.1410
      Credit_amount  num    0.1087    0.0955    0.8784
   Installment_rate  num    0.2187    0.1225    0.5600
                Age  num    0.1043    0.0972    0.9316
mean SE ratio stump/logit=0.906 (cat=0.916, num=0.878)



=== German performance (100 splits) ===
logit-WoE  AUC=0.8129±0.0224 | stump-boost AUC=0.7912±0.0251
mean AUC diff (stump-logit)=-0.02169 | Wilcoxon p=0.0000 | DeLong frac p<0.05=0.72



Saved figures: 05_german_s_vs_woe, 05_german_beta, 05_german_auc_box
Saved tables : 05_german_iv, 05_german_isomorphism, 05_german_se, 05_german_perf

================ GENERALIZATION HEADLINE ================
GERMAN (1000 rows, 30% bad, 11 nominal + 4 numeric kept):
  isomorphism: mean beta_eff=1.000, mean w_corr(s,WoE)=1.000
  SE alignment: mean ratio stump/logit=0.906
  performance : logit AUC=0.813 vs stump AUC=0.791 (Wilcoxon p=0.000)
Compare GMSC (02/03/04): beta_eff~1.00, iso_nomiss SE ratio~1.20, stump~logit AUC.

05 complete.
